# M5 — Bằng chứng mùa vụ trên tập train
Notebook độc lập làm cơ sở cho feature engineering trong `tft_m5_zero_test_improved.ipynb`. Chỉ phân tích đến cutoff train; không đọc sales validation/test để chọn chu kỳ.

**Câu hỏi:** mùa vụ tuần/năm xuất hiện ở đâu, ổn định đến mức nào, và hỗ trợ lựa chọn feature nào? Các chỉ số dưới đây là bằng chứng mô tả, không phải kiểm định nhân quả hay chứng minh hiệu quả dự báo. Tổng có mùa vụ không có nghĩa mọi item–store đều có.

Chạy lần lượt các cell. Dependency: `numpy pandas scipy statsmodels matplotlib`. Mặc định dùng full M5; đặt `M5_SYNTHETIC=1` để kiểm tra pipeline bằng dữ liệu giả (không dùng kết quả này làm bằng chứng M5). Không cần PyTorch hoặc train TFT.

In [ ]:
import os, json, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import periodogram, detrend
from statsmodels.tsa.stattools import acf
from statsmodels.tsa.seasonal import STL

SEED = 42
TRAIN_END = int(os.getenv('M5_TRAIN_END', '1885'))
SYNTHETIC = os.getenv('M5_SYNTHETIC', '0') == '1'
SAMPLE_SERIES = int(os.getenv('M5_SAMPLE_SERIES', '0'))  # 0 = full; >0 = subset
L12_PER_STRATUM = int(os.getenv('M5_L12_PER_STRATUM', '4'))
MIN_ACTIVE_DAYS = 365
OUT_DIR = Path(os.getenv('M5_SEASONALITY_OUT', 'seasonality_outputs'))
OUT_DIR.mkdir(parents=True, exist_ok=True)
assert TRAIN_END >= 730 and SAMPLE_SERIES >= 0 and L12_PER_STRATUM > 0
rng = np.random.default_rng(SEED)
print('TRAIN_END:', TRAIN_END, '| synthetic:', SYNTHETIC, '| subset:', SAMPLE_SERIES or 'full')


## 1. Đọc dữ liệu và audit
Sales chỉ đọc các cột đến cutoff. Giá chỉ dùng thống kê độ phủ quan sát, không suy diễn tồn kho. Sample là mẫu ngẫu nhiên cố định; tổng trong sample không phải tổng M5 đầy đủ. Không tự thay NaN sales thành zero trong EDA: dữ liệu lỗi cần được xử lý rõ ràng.

In [ ]:
META = ['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id']
DAYS = [f'd_{d}' for d in range(1, TRAIN_END + 1)]
DATA_ROOT = None
if SYNTHETIC:
    n = 120
    dates = pd.date_range('2011-01-29', periods=TRAIN_END)
    t = np.arange(TRAIN_END)
    cal = pd.DataFrame({'date': dates, 'd': DAYS, 'wm_yr_wk': t // 7,
                        'event_name_1': np.where(t % 91 == 0, 'synthetic_event', None),
                        'event_name_2': None})
    meta = pd.DataFrame({'id': [f's{i}' for i in range(n)], 'item_id': [f'i{i}' for i in range(n)],
                         'dept_id': [f'd{i%3}' for i in range(n)], 'cat_id': 'c0',
                         'store_id': [f'st{i%6}' for i in range(n)],
                         'state_id': [f'state{i%2}' for i in range(n)]})
    rate = rng.uniform(.1, 8, (n, 1)) * (1 + .35*np.cos(2*np.pi*t/7))
    rate = rate * (1 + .2*np.sin(2*np.pi*t/365.25))
    sales = rng.poisson(rate).astype(np.float32)
    coverage = np.full(TRAIN_END, n)
else:
    candidates = [Path(os.environ['M5_DATA_ROOT'])] if os.getenv('M5_DATA_ROOT') else []
    candidates += [Path('/kaggle/input/m5-forecasting-accuracy'), Path('../data/raw'), Path('data/raw'), Path('../../data/raw')]
    if Path('/kaggle/input').exists():
        candidates += sorted(p.parent for p in Path('/kaggle/input').rglob('calendar.csv'))
    for root in candidates:
        if root.is_file():
            root = root.parent
        files = [root / 'sales_train_evaluation.csv', root / 'sales_train_validation.csv']
        available = [p for p in files if p.exists()]
        if available and (root/'calendar.csv').exists() and (root/'sell_prices.csv').exists():
            DATA_ROOT, sales_file = root.resolve(), available[0]
            break
    if DATA_ROOT is None:
        raise FileNotFoundError('Đặt M5_DATA_ROOT tới thư mục chứa sales, calendar.csv và sell_prices.csv.')
    frame = pd.read_csv(sales_file, usecols=META + DAYS, dtype={d: 'float32' for d in DAYS})
    if SAMPLE_SERIES and SAMPLE_SERIES < len(frame):
        frame = frame.sample(SAMPLE_SERIES, random_state=SEED).reset_index(drop=True)
    meta = frame[META].copy()
    sales = frame[DAYS].to_numpy(dtype=np.float32, copy=True)
    del frame
    calendar = pd.read_csv(DATA_ROOT/'calendar.csv')
    calendar['d_num'] = calendar['d'].str[2:].astype(int)
    cal = calendar.set_index('d_num').loc[np.arange(1, TRAIN_END+1)].reset_index()
    cal['date'] = pd.to_datetime(cal['date'])
    prices = pd.read_csv(DATA_ROOT/'sell_prices.csv', usecols=['store_id','item_id','wm_yr_wk','sell_price'])
    prices = prices.merge(meta[['store_id','item_id']].drop_duplicates(), on=['store_id','item_id'], how='inner')
    prices = prices[np.isfinite(prices['sell_price'])].drop_duplicates(['store_id','item_id','wm_yr_wk'])
    counts = prices.groupby('wm_yr_wk').size()
    coverage = cal['wm_yr_wk'].map(counts).fillna(0).to_numpy()
    del prices, calendar

dates = pd.DatetimeIndex(cal['date'])
assert meta['id'].is_unique, 'Trùng series ID'
assert len(dates) == TRAIN_END and dates.is_unique and dates.is_monotonic_increasing
assert (np.diff(dates.values) == np.timedelta64(1, 'D')).all(), 'Lịch không liên tục'
assert np.isfinite(sales).all() and (sales >= 0).all(), 'Sales thiếu, âm hoặc không hữu hạn'
event = cal[['event_name_1','event_name_2']].notna().any(axis=1).to_numpy()
positive = sales > 0
first = np.where(positive.any(axis=1), positive.argmax(axis=1), -1)
series_stats = meta.copy()
series_stats['zero_rate'] = (~positive).mean(axis=1)
series_stats['mean_sales'] = sales.mean(axis=1)
series_stats['first_sale_index'] = first
series_stats['active_days'] = np.where(first >= 0, TRAIN_END-first, 0)
series_stats.to_csv(OUT_DIR/'train_series_audit.csv', index=False)
audit = {'synthetic': SYNTHETIC, 'data_root': str(DATA_ROOT), 'train_end': TRAIN_END,
         'start_date': str(dates.min().date()), 'end_date': str(dates.max().date()),
         'n_series': len(meta), 'sample_series_requested': SAMPLE_SERIES,
         'zero_rate': float((~positive).mean()), 'seed': SEED}
pd.DataFrame([audit]).to_csv(OUT_DIR/'train_data_audit.csv', index=False)
print(audit)
fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
axes[0].plot(dates, sales.sum(axis=0, dtype=np.float64)); axes[0].set_ylabel('Total units')
axes[1].plot(dates, coverage, label='Observed price pairs')
axes[1].plot(dates, np.cumsum(np.bincount(first[first >= 0], minlength=TRAIN_END)), label='Ever sold')
axes[1].legend(); axes[1].set_ylabel('Series count')
fig.tight_layout(); fig.savefig(OUT_DIR/'train_coverage.png', dpi=130); plt.show()
del positive


## 2. Phân cấp và chọn mẫu L12
Phân tích toàn bộ L1/L2/L3/L9. L12 lấy mẫu theo 3 mức bán × 3 mức zero-rate (xếp hạng để xử lý ties), kèm chuỗi toàn zero nếu có. Cắt đoạn trước lần bán đầu khi tính chỉ số; không cắt các zero nằm giữa lịch sử. Chuỗi có lịch sử hoạt động dưới 365 ngày hoặc hằng được đánh dấu không đủ điều kiện.

In [ ]:
PANELS = {}
for level, columns in [('L1', []), ('L2', ['state_id']), ('L3', ['store_id']), ('L9', ['store_id','dept_id'])]:
    keys = meta[columns].astype(str).agg('|'.join, axis=1) if columns else pd.Series('TOTAL', index=meta.index)
    codes, labels = pd.factorize(keys, sort=True)
    grouped = np.zeros((len(labels), TRAIN_END), dtype=np.float64)
    np.add.at(grouped, codes, sales)
    PANELS[level] = (list(labels), grouped)
series_stats['volume_bin'] = pd.qcut(series_stats['mean_sales'].rank(method='first'), 3, labels=['low','mid','high'])
series_stats['zero_bin'] = pd.qcut(series_stats['zero_rate'].rank(method='first'), 3, labels=['low','mid','high'])
selected = []
for _, group in series_stats.groupby(['volume_bin','zero_bin'], observed=True):
    selected.extend(group.sample(min(L12_PER_STRATUM, len(group)), random_state=SEED).index.tolist())
all_zero = series_stats.index[series_stats['zero_rate'].eq(1)].tolist()
selected = sorted(set(selected + all_zero[:L12_PER_STRATUM]))
PANELS['L12'] = (meta.loc[selected, 'id'].tolist(), sales[selected].astype(np.float64))
manifest = []
for level, (labels, values) in PANELS.items():
    for label, y in zip(labels, values):
        nz = np.flatnonzero(y > 0)
        start = int(nz[0]) if len(nz) else len(y)
        active = y[start:]
        reason = 'all_zero' if not len(active) else ('short_history' if len(active)<MIN_ACTIVE_DAYS else ('constant' if np.ptp(active)==0 else 'eligible'))
        manifest.append({'level': level, 'group': label, 'start_index': start, 'active_days': len(active),
                         'zero_rate': float(np.mean(y == 0)), 'eligibility': reason})
manifest = pd.DataFrame(manifest)
manifest.to_csv(OUT_DIR/'analysis_series_manifest.csv', index=False)
series_stats.loc[selected].to_csv(OUT_DIR/'l12_sampling_manifest.csv', index=False)
print(manifest.groupby(['level','eligibility']).size().to_string())


## 3. Định lượng mùa vụ tuần và năm
**Tuần:** chuẩn hóa trong từng tuần Mon–Sun đầy đủ (bỏ tuần tổng bằng 0), lấy profile theo thứ; so độ tương đồng profile giữa các năm. ACF và phổ tính sau linear detrend. STL robust chu kỳ 7 cung cấp chỉ số strength, không phải p-value.

**Năm:** hồi quy trên `log1p(sales)` với baseline gồm spline trend tuyến tính từng đoạn (knots mỗi 730 ngày), weekday và cờ event. So với baseline + một cặp Fourier năm, dùng expanding-year holdouts nằm hoàn toàn trong train (ít nhất 2 năm lịch đầy đủ trước holdout). Gain là giảm SSE ngoài mẫu; có thể âm. Profile tháng chỉ dùng năm lịch đầy đủ, chuẩn hóa mức bán trong năm.

Giới hạn: trend specification và event flag giản lược có thể ảnh hưởng kết quả; yearly gain không loại bỏ hoàn toàn confounding. Lag 364 còn mang hiệu ứng tuần. Các nhãn tổng hợp phía sau là heuristic minh bạch, không phải kiểm định ý nghĩa thống kê.

In [ ]:
def full_years(ds):
    return [int(year) for year in np.unique(ds.year)
            if len(ds[ds.year == year]) == (366 if pd.Timestamp(int(year),12,31).is_leap_year else 365)]

def weekly_profiles(y, ds):
    f = pd.DataFrame({'y': y, 'date': ds, 'dow': ds.dayofweek})
    f['week'] = ds - pd.to_timedelta(ds.dayofweek, unit='D')
    by = f.groupby('week')['y']
    f['week_total'] = by.transform('sum')
    f = f[(by.transform('size') == 7) & (f['week_total'] > 0)].copy()
    f['relative'] = f['y'] / (f['week_total']/7)
    f['year'] = f['date'].dt.year
    return f, f.groupby('dow')['relative'].mean().reindex(range(7))

def year_design(ds):
    t = (ds - dates[0]).days.to_numpy(dtype=float)
    parts = [np.ones(len(ds)), t/365.25]
    parts += [np.maximum(t-k, 0)/365.25 for k in range(730, TRAIN_END, 730)]
    parts += [(ds.dayofweek == d).astype(float) for d in range(1,7)]
    pos = dates.get_indexer(ds)
    parts += [event[pos].astype(float)]
    base = np.column_stack(parts)
    annual = np.column_stack([np.sin(2*np.pi*t/365.25), np.cos(2*np.pi*t/365.25)])
    return base, np.column_stack([base, annual])

def annual_cv(y, ds):
    complete = full_years(ds)
    folds = []
    b, a = year_design(ds)
    z = np.log1p(y)
    for year in complete:
        if sum(prior < year for prior in complete) < 2:
            continue
        train, test = ds.year < year, ds.year == year
        errors = []
        for x in (b, a):
            coef = np.linalg.lstsq(x[train], z[train], rcond=None)[0]
            errors.append(float(np.sum((z[test] - x[test] @ coef)**2)))
        folds.append({'year': year, 'base_sse': errors[0], 'annual_sse': errors[1]})
    if not folds:
        return np.nan, []
    denominator = sum(f['base_sse'] for f in folds)
    return (1-sum(f['annual_sse'] for f in folds)/denominator if denominator > 1e-12 else np.nan), folds

weekly_rows, annual_rows, fold_rows = [], [], []
for row in manifest.to_dict('records'):
    level, label, start = row['level'], row['group'], row['start_index']
    if row['eligibility'] != 'eligible':
        continue
    labels, values = PANELS[level]
    y, ds = values[labels.index(label), start:], dates[start:]
    residual = detrend(y, type='linear')
    correlations = acf(residual, nlags=min(366, len(y)-1), fft=True)
    freq, power = periodogram(residual)
    band = (freq >= 1/7 - .005) & (freq <= 1/7 + .005)
    spectral_share = float(power[band].sum()/max(power[1:].sum(), 1e-12))
    profile_data, profile = weekly_profiles(y, ds)
    profiles = profile_data.groupby(['year','dow'])['relative'].mean().unstack('dow').reindex(columns=range(7))
    profiles = profiles.loc[profile_data.groupby('year')['week'].nunique().ge(26)]
    similarities = []
    for i in range(len(profiles)):
        for j in range(i):
            u, v = profiles.iloc[i].to_numpy(), profiles.iloc[j].to_numpy()
            if np.isfinite(u).all() and np.isfinite(v).all() and min(np.std(u),np.std(v))>1e-9:
                similarities.append(float(np.corrcoef(u,v)[0,1]))
    decomposition = STL(y, period=7, robust=True).fit()
    denominator = np.var(decomposition.seasonal+decomposition.resid)
    strength = max(0., 1-np.var(decomposition.resid)/denominator) if denominator > 1e-12 else np.nan
    non_event_profile = profile_data.loc[~event[dates.get_indexer(pd.DatetimeIndex(profile_data['date']))]].groupby('dow')['relative'].mean().reindex(range(7))
    weekly_rows.append({'level':level, 'group':label, 'active_days':len(y),
        'acf_7':correlations[7], 'acf_14':correlations[14], 'acf_28':correlations[28],
        'weekly_spectral_share':spectral_share, 'stl_weekly_strength':strength,
        'weekday_peak_to_trough':float(profile.max()-profile.min()),
        'non_event_peak_to_trough':float(non_event_profile.max()-non_event_profile.min()),
        'profile_year_pairs':len(similarities),
        'profile_year_correlation':float(np.median(similarities)) if similarities else np.nan})
    gain, folds = annual_cv(y, ds)
    for fold in folds:
        fold_rows.append({'level':level,'group':label,**fold})
    annual_rows.append({'level':level,'group':label,'complete_years':len(full_years(ds)),
        'annual_cv_folds':len(folds),'annual_cv_sse_gain':gain,
        'annual_positive_fold_fraction':np.mean([f['annual_sse']<f['base_sse'] for f in folds]) if folds else np.nan,
        **{f'acf_{lag}':correlations[lag] if lag<len(correlations) else np.nan for lag in [364,365,366]}})
weekly = pd.DataFrame(weekly_rows)
annual = pd.DataFrame(annual_rows)
if weekly.empty:
    raise ValueError('Không có chuỗi đủ điều kiện; xem analysis_series_manifest.csv.')
weekly.to_csv(OUT_DIR/'weekly_evidence.csv', index=False)
annual.to_csv(OUT_DIR/'annual_evidence.csv', index=False)
pd.DataFrame(fold_rows, columns=['level','group','year','base_sse','annual_sse']).to_csv(OUT_DIR/'annual_cv_folds.csv', index=False)
print(weekly.groupby('level')[['acf_7','stl_weekly_strength','profile_year_correlation']].median().round(3))
print(annual.groupby('level')[['annual_cv_folds','annual_cv_sse_gain']].median().round(3))


## 4. Biểu đồ đại diện
Chọn nhóm có mức bán trung vị ở mỗi cấp để tránh chỉ chọn đồ thị thể hiện mùa vụ đẹp nhất. Các bảng CSV phía trên vẫn chứa toàn bộ nhóm đủ điều kiện. Khoảng lag quanh năm hiển thị riêng để tránh nhầm lag 364 với bằng chứng năm độc lập.

In [ ]:
representatives = []
for level, (labels, values) in PANELS.items():
    eligible = manifest[(manifest['level']==level) & manifest['eligibility'].eq('eligible')]['group'].tolist()
    if not eligible:
        continue
    indices = [labels.index(k) for k in eligible]
    ordered = sorted(indices, key=lambda i: values[i].mean())
    i = ordered[len(ordered)//2]
    label = labels[i]
    start = int(manifest.loc[(manifest['level']==level)&manifest['group'].eq(label),'start_index'].iloc[0])
    y, ds = values[i,start:], dates[start:]
    representatives.append({'level':level,'group':label})
    f, profile = weekly_profiles(y, ds)
    residual = detrend(y)
    correlations = acf(residual, nlags=min(400,len(y)-1), fft=True)
    freq, power = periodogram(residual)
    decomposition = STL(y, period=7, robust=True).fit()
    fig, axes = plt.subplots(3,2, figsize=(14,11))
    axes[0,0].plot(ds,y,alpha=.5,label='Sales'); axes[0,0].plot(ds,decomposition.trend,label='STL trend'); axes[0,0].legend()
    for year, g in f.groupby('year'):
        if g['week'].nunique() >= 26:
            axes[0,1].plot(range(7),g.groupby('dow')['relative'].mean().reindex(range(7)),label=str(year))
    axes[0,1].set_xticks(range(7), ['Mon','Tue','Wed','Thu','Fri','Sat','Sun']); axes[0,1].legend(fontsize=8)
    axes[0,1].set_title('Within-week normalized profiles by year')
    axes[1,0].stem(np.arange(min(57,len(correlations))),correlations[:57]); axes[1,0].set_title('ACF after linear detrend (no IID confidence band)')
    axes[1,1].plot(freq[1:],power[1:]); axes[1,1].axvline(1/7,color='r',linestyle='--'); axes[1,1].set_title('Periodogram; red = 1/7 per day')
    for year in full_years(ds):
        mask = ds.year==year
        z = pd.Series(y[mask],index=ds[mask])
        if z.mean()>0:
            axes[2,0].plot(range(1,13),z.groupby(z.index.month).mean().reindex(range(1,13))/z.mean(),label=str(year))
    axes[2,0].legend(fontsize=8); axes[2,0].set_title('Monthly means / yearly mean; full years only')
    lags = np.arange(350,min(381,len(correlations)))
    axes[2,1].stem(lags,correlations[lags]); axes[2,1].set_title('ACF near annual lag; 364 also retains weekday')
    fig.suptitle(f'{level} | {label}' + (' | SYNTHETIC' if SYNTHETIC else ''))
    fig.tight_layout(); fig.savefig(OUT_DIR/f'{level}_representative.png',dpi=130); plt.show()
    fig, axes = plt.subplots(2,1,figsize=(12,5),sharex=True)
    tail = min(112,len(y))
    axes[0].plot(ds[-tail:],decomposition.seasonal[-tail:]); axes[0].set_ylabel('STL weekly')
    axes[1].plot(ds[-tail:],decomposition.resid[-tail:]); axes[1].set_ylabel('Residual')
    fig.suptitle(f'{level} | {label} | Last 112 train days')
    fig.tight_layout(); fig.savefig(OUT_DIR/f'{level}_stl.png',dpi=130); plt.show()
pd.DataFrame(representatives).to_csv(OUT_DIR/'plot_representatives.csv',index=False)


## 5. Tổng hợp bằng chứng và quyết định feature
Heuristic tuần: ACF7 > 0.1, spectral share > 0.05, STL strength > 0.1, median tương quan profile giữa năm > 0.5. Cần ít nhất 3/4 dấu hiệu để gắn nhãn `converging_descriptive_evidence`; ngưỡng không phải mức ý nghĩa thống kê.

Năm: cần ít nhất 2 holdout years, tổng SSE giảm > 1% và phần lớn fold có cải thiện; vẫn phải đọc đồ thị và kiểm tra độ nhạy với trend/event trước khi kết luận. Nhãn không tự động bật/tắt feature của TFT. Không có p-value hay tuyên bố significance; nếu bổ sung kiểm định nhiều nhóm cần block bootstrap và điều chỉnh đa kiểm định.

In [ ]:
evidence = weekly.merge(annual, on=['level','group'], validate='one_to_one')
evidence['weekly_signal_count'] = sum((evidence[col] > threshold).astype(int) for col, threshold in [('acf_7',.1),('weekly_spectral_share',.05),('stl_weekly_strength',.1),('profile_year_correlation',.5)])
evidence['weekly_assessment'] = np.where(evidence['weekly_signal_count']>=3, 'converging_descriptive_evidence','mixed_or_weak_evidence')
evidence['annual_assessment'] = np.select([evidence['annual_cv_folds']<2,(evidence['annual_cv_sse_gain']>.01)&(evidence['annual_positive_fold_fraction']>.5)], ['insufficient_yearly_holdouts','provisional_incremental_annual_evidence'], default='mixed_or_no_incremental_gain')
evidence.to_csv(OUT_DIR/'seasonality_evidence_summary.csv',index=False)
summary = evidence.groupby('level').agg(n_analyzed=('group','size'), median_acf7=('acf_7','median'), median_weekly_strength=('stl_weekly_strength','median'), median_annual_gain=('annual_cv_sse_gain','median'))
print(summary.round(3).to_string())
print(evidence.groupby(['level','weekly_assessment']).size().to_string())
print(evidence.groupby(['level','annual_assessment']).size().to_string())
report = ['# Seasonality evidence report', '**SYNTHETIC: pipeline verification only.**' if SYNTHETIC else '**Train-only M5 analysis.**', f'Cutoff d_{TRAIN_END}; series {len(meta)}; subset request {SAMPLE_SERIES}.', 'L12 is a stratified sample. Subset aggregates are not full M5 totals.', '## Summary', '```text'+chr(10)+summary.round(4).to_string()+chr(10)+'```', '## Interpretation', 'Read CSV evidence and representative plots together. Labels are descriptive heuristics, not significance tests.', 'Weekly candidates: sin_wday/cos_wday/is_weekend and lag_7/14/28/35/56. Annual candidates: sin_month/cos_month, sin_year/cos_year and lag_364.', 'Annual evidence is provisional: few years, trend/event confounding; lag 364 also preserves weekday. Monthly profiles do not prove a 30-day cycle.', 'Price coverage is not inventory availability. TFT improved uses 51 past, 10 future and 5 static inputs.', '## Next experiment', 'Compare full TFT against removing weekly calendar, weekly lags and annual features separately. Match series, origins, seeds, loss, budget and early stopping. Evaluate uncalibrated BU WRMSSE-5 and per-level metrics on validation.', 'Rolling/hierarchy features retain seasonal information: ablations measure incremental feature contributions. Do not select features on test.']
(OUT_DIR/'seasonality_report.md').write_text((chr(10)*2).join(report),encoding='utf-8')
config = {**audit, 'l12_per_stratum':L12_PER_STRATUM, 'min_active_days':MIN_ACTIVE_DAYS, 'annual_model':'log1p; piecewise linear trend knots 730d; weekday; event; annual Fourier K=1', 'weekly_heuristic_thresholds':{'acf7':.1,'spectral_share':.05,'stl_strength':.1,'profile_corr':.5}}
(OUT_DIR/'analysis_config.json').write_text(json.dumps(config,ensure_ascii=False,indent=2),encoding='utf-8')
print('Saved:',OUT_DIR.resolve())
